In [7]:
!pip install qrcode pillow pandas requests

In [ ]:
import pandas as pd
import qrcode
from pathlib import Path

# --- CONFIGURATION ---
SHEET_ID = "Insert Sheet_ID Here"
CSV_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"
SHEET_FILENAME = "guests_data.csv"
OUTPUT_FOLDER = "QR code data"

# Prepare output directory
output_dir = Path(OUTPUT_FOLDER)
output_dir.mkdir(parents=True, exist_ok=True)

# Fetch latest sheet data directly over HTTP
print("Fetching live guest list from Google Sheets...")
try:
    df = pd.read_csv(CSV_URL)
    print(f"✅ Successfully loaded {len(df)} records from Google Sheets!")
except Exception as e:
    raise ConnectionError(f"Failed to fetch Google Sheet. Check Sheet ID and sharing permissions. Details: {e}")

df.head()

Fetching live guest list from Google Sheets...


✅ Successfully loaded 2795 records from Google Sheets!


,Name,Email,phone,national_id,Do you need transportation (Bus)?,UUID,Attended,Timestamp,email_sent
0,Mahmoud Mohamed Ali Younes,mahmoudaboyounes6@gmail.com,1024857119,30601012718894,yes,USR-5058305331-MLVLTGTTWT,NaN,NaN,True
1,عبدالغني عمر,ao146918@gmail.com,01554672709,30401152102274,no,USR-6703697958-BZEGYXNZJJ,NaN,NaN,True
2,Amr Ahmed Hassan,amrahmedhassan841@gmail.com,01033934753,30412192500436,yes,USR-9411562244-MCJEBIMUPG,NaN,NaN,True
3,ahmed mohamed ahmed shalaby,ahmedmohamadshalaby@gmail.com,01065785588,30611301900175,no,USR-6866550076-YRHNIWKNYK,NaN,NaN,True
4,هانم رفعت عابدين عباس,hanarefaat13488@gmail.com,01101334732,28804130104506,yes,USR-1836260111-LAHCPRKRHV,NaN,NaN,True


In [9]:
# Locate UUID column dynamically regardless of casing
uuid_col = next((col for col in df.columns if col.strip().lower() == 'uuid'), None)

if not uuid_col:
    raise KeyError(f"Could not find 'UUID' column. Available columns: {list(df.columns)}")

generated_count = 0

for index, row in df.iterrows():
    raw_uuid = str(row[uuid_col]).strip()
    
    if not raw_uuid or raw_uuid.lower() == 'nan':
        continue
    
    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=10,
        border=4,
    )
    qr.add_data(raw_uuid)
    qr.make(fit=True)
    
    img = qr.make_image(fill_color="black", back_color="white")
    
    file_path = output_dir / f"{raw_uuid}.png"
    img.save(file_path)
    generated_count += 1

# Save local CSV copy for email sender script
df.to_csv(SHEET_FILENAME, index=False)

print(f"✅ Generated {generated_count} QR codes and cached data to '{SHEET_FILENAME}'.")

✅ Generated 2795 QR codes and cached data to 'guests_data.csv'.
